# Sheet 9 — Research sandbox (Chapter 9)

Open-ended starting points. None has a known complete answer in the literature surveyed in the report.

In [1]:
# Setup: make the repository importable whether or not `pip install -e .` was run
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, networkx as nx, matplotlib.pyplot as plt
from tonnetzkit import *
%matplotlib inline
import collections, random

### Project A — Minimal edge Tonnetz
Among all closed edge gluings of the 12 whole-tone shell chords, which surfaces occur, and with what exact counts?
Random sampling (Sheet 6) is a start; an exact enumeration needs symmetry reduction.

In [2]:
odd = [1, 3, 5, 7, 9, 11]
faces = [tuple(sorted(essential_notes(r, 'dom7'))) for r in odd] + [tuple(sorted(essential_notes(r, 'maj9'))) for r in odd]
# number of gluing choices: each note labels 6 edge-slots -> 15 perfect matchings x 2^3 orientations
print('size of the search space: (15 * 8)^6 =', (15*8)**6)

size of the search space: (15 * 8)^6 = 2985984000000


### Project B — Hamiltonian cycles for seventh chords
Build a graph on seventh chords whose edges are single-semitone or whole-tone moves (cf. Cannas & Andreatta 2018) and
count Hamiltonian cycles. Warning: the backtracking in `hamiltonian_cycles` is exponential; add pruning (e.g. degree-2 forcing).

In [3]:
SEV = {'dom7': (0,4,7,10), 'min7': (0,3,7,10), 'hdim7': (0,3,6,10), 'maj7': (0,4,7,11)}
chords = [frozenset((r+i) % 12 for i in iv) for r in range(12) for iv in SEV.values()]
chords = list(dict.fromkeys(chords))
G7 = nx.Graph()
for a in chords:
    for b in chords:
        if a != b and len(a & b) == 3:   # share three of four notes
            G7.add_edge(a, b)
print(G7.number_of_nodes(), 'chords;', G7.number_of_edges(), 'edges; degrees', collections.Counter(dict(G7.degree()).values()))

48 chords; 108 edges; degrees Counter({4: 36, 6: 12})


### Project C — Perceptual test
Generate stimuli for a listening test comparing Hamiltonian cycles with matched random walks; export WAV files.

In [4]:
G = plr_graph(); H = hamiltonian_cycles(G)
def random_walk(n=24, seed=0):
    rng = random.Random(seed); t = Triad.parse('C'); out = [t]
    for _ in range(n-1):
        t = rng.choice(list(G.neighbors(t))); out.append(t)
    return out
print('Hamiltonian :', ' '.join(map(str, H[0][:10])), '...')
print('random walk :', ' '.join(map(str, random_walk()[:10])), '...')

Hamiltonian : C Cm D# D#m F# F#m A C#m E Em ...
random walk : C Em E Em E C#m E G#m G# Cm ...
